In [ ]:
import logging
import sys
import datetime

In [ ]:
figures = "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v1/figures"
obj = "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v1/objects"

# CellphoneDB 

## Download database from source

### Display database versions

In [ ]:
from IPython.display import HTML, display
from cellphonedb.utils import db_releases_utils
import pandas as pd
import glob
import os

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
display(HTML(db_releases_utils.get_remote_database_versions_html()['db_releases_html_table']))

### Define the version and the path to download database

In [ ]:
# -- Version of the databse
cpdb_version = 'v5.0.0'

# -- Path where the input files to generate the database are located
cpdb_target_dir = os.path.join(f'{obj}', 'cellphoneDB_db',cpdb_version)

In [ ]:
cpdb_target_dir

### Download database

In [ ]:
from cellphonedb.utils import db_utils

db_utils.download_database(cpdb_target_dir, cpdb_version)

## How to run CellphoneDB for the statistical method

### Input files

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
from anndata import AnnData
import pathlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import sys

In [ ]:
from matplotlib.pyplot import MultipleLocator

In [ ]:
# load the samples_all_annotation_diseased.h5ad with count layer
samples_all_diseased = sc.read_h5ad(f'{obj}/samples_all_annotation_diseased.h5ad')

# load the samples_all_annotation_healthy.h5ad with count layer
samples_all_healthy = sc.read_h5ad(f'{obj}/samples_all_annotation_healthy.h5ad')


In [ ]:
# get the normalised_log_counts
sc.pp.normalize_total(samples_all_diseased)
sc.pp.log1p(samples_all_diseased)

sc.pp.normalize_total(samples_all_healthy)
sc.pp.log1p(samples_all_healthy)

In [ ]:
# split the T_B merged adata into young and old objects
T_B_clusters = ['CCR7_CD4_Tnaive', 'CXCR6_CD4_Trm', 'CXCL13_CD4_Tex', 'FOXP3_CD4_Treg',
                'IFITM3_CD8_Teffector', 'GZMB_CD8_Teffector', 'GZMK_CD8_Tem', 'ZNF683_CD8_Trm', 'CXCL13_CD8_Tex', 
                'Naive_B', 'Memory_B']

# diseased
samples_all_diseased_T_B = samples_all_diseased[samples_all_diseased.obs['sub_anno'].isin(T_B_clusters)].copy()
samples_all_diseased_T_B_young = samples_all_diseased_T_B[samples_all_diseased_T_B.obs['Age_type'] == 'Young'].copy()
samples_all_diseased_T_B_old = samples_all_diseased_T_B[samples_all_diseased_T_B.obs['Age_type'] == 'Old'].copy()

#healthy
samples_all_healthy_T_B = samples_all_healthy[samples_all_healthy.obs['sub_anno'].isin(T_B_clusters)].copy()
samples_all_healthy_T_B_young = samples_all_healthy_T_B[samples_all_healthy_T_B.obs['Age_type'] == 'Young'].copy()
samples_all_healthy_T_B_old = samples_all_healthy_T_B[samples_all_healthy_T_B.obs['Age_type'] == 'Old'].copy()


In [ ]:
AnnData.write_h5ad(samples_all_diseased_T_B_young, filename=f'{obj}/cellphoneDB_db/samples_all_diseased_T_B_young_normalised_log_counts.h5ad')
AnnData.write_h5ad(samples_all_diseased_T_B_old, filename=f'{obj}/cellphoneDB_db/samples_all_diseased_T_B_old_normalised_log_counts.h5ad')

AnnData.write_h5ad(samples_all_healthy_T_B_young, filename=f'{obj}/cellphoneDB_db/samples_all_healthy_T_B_young_normalised_log_counts.h5ad')
AnnData.write_h5ad(samples_all_healthy_T_B_old, filename=f'{obj}/cellphoneDB_db/samples_all_healthy_T_B_old_normalised_log_counts.h5ad')

In [ ]:
# get the meta_file (sub_anno)

#diseased
samples_all_diseased_T_B_young_subAnno_meta_file = samples_all_diseased_T_B_young.obs['Age_type'].str.cat(samples_all_diseased_T_B_young.obs['sub_anno'], sep='_')
samples_all_diseased_T_B_young_subAnno_meta_file = samples_all_diseased_T_B_young_subAnno_meta_file.to_frame().rename_axis('barcode_sample').reset_index().rename(columns = {'Age_type' : 'cell_type'})

samples_all_diseased_T_B_old_subAnno_meta_file = samples_all_diseased_T_B_old.obs['Age_type'].str.cat(samples_all_diseased_T_B_old.obs['sub_anno'], sep='_')
samples_all_diseased_T_B_old_subAnno_meta_file = samples_all_diseased_T_B_old_subAnno_meta_file.to_frame().rename_axis('barcode_sample').reset_index().rename(columns = {'Age_type' : 'cell_type'})


#healthy
samples_all_healthy_T_B_young_subAnno_meta_file = samples_all_healthy_T_B_young.obs['Age_type'].str.cat(samples_all_healthy_T_B_young.obs['sub_anno'], sep='_')
samples_all_healthy_T_B_young_subAnno_meta_file = samples_all_healthy_T_B_young_subAnno_meta_file.to_frame().rename_axis('barcode_sample').reset_index().rename(columns = {'Age_type' : 'cell_type'})

samples_all_healthy_T_B_old_subAnno_meta_file = samples_all_healthy_T_B_old.obs['Age_type'].str.cat(samples_all_healthy_T_B_old.obs['sub_anno'], sep='_')
samples_all_healthy_T_B_old_subAnno_meta_file = samples_all_healthy_T_B_old_subAnno_meta_file.to_frame().rename_axis('barcode_sample').reset_index().rename(columns = {'Age_type' : 'cell_type'})

In [ ]:
# Check barcodes in metadata and counts are the same
print(
list(samples_all_diseased_T_B_young.obs.index).sort() == list(samples_all_diseased_T_B_young_subAnno_meta_file['barcode_sample']).sort(),
list(samples_all_diseased_T_B_old.obs.index).sort() == list(samples_all_diseased_T_B_old_subAnno_meta_file['barcode_sample']).sort())

In [ ]:
# Check barcodes in metadata and counts are the same
print(
list(samples_all_healthy_T_B_young.obs.index).sort() == list(samples_all_healthy_T_B_young_subAnno_meta_file['barcode_sample']).sort(),
list(samples_all_healthy_T_B_old.obs.index).sort() == list(samples_all_healthy_T_B_old_subAnno_meta_file['barcode_sample']).sort())

In [ ]:
#save meta files

##subIdentity diseased
samples_all_diseased_T_B_young_subAnno_meta_file.to_csv(f'{obj}/cellphoneDB_db/samples_all_diseased_T_B_young_subAnno_meta_file.tsv', sep = '\t', index=False)
samples_all_diseased_T_B_old_subAnno_meta_file.to_csv(f'{obj}/cellphoneDB_db/samples_all_diseased_T_B_old_subAnno_meta_file.tsv', sep = '\t', index=False)

##subIdentity healthy
samples_all_healthy_T_B_young_subAnno_meta_file.to_csv(f'{obj}/cellphoneDB_db/samples_all_healthy_T_B_young_subAnno_meta_file.tsv', sep = '\t', index=False)
samples_all_healthy_T_B_old_subAnno_meta_file.to_csv(f'{obj}/cellphoneDB_db/samples_all_healthy_T_B_old_subAnno_meta_file.tsv', sep = '\t', index=False)




In [ ]:
cpdb_file_path = f'{obj}/cellphoneDB_db/v5.0.0/cellphonedb.zip'

#diseased
diseased_T_B_young_subAnno_meta_file_path = f'{obj}/cellphoneDB_db/samples_all_diseased_T_B_young_subAnno_meta_file.tsv'
diseased_T_B_young_degs_file_path = f'{obj}/cellphoneDB_db/diseased_T_B_young_subAnno_DEGs_file.tsv'
diseased_T_B_young_counts_file_path = f'{obj}/cellphoneDB_db/samples_all_diseased_T_B_young_normalised_log_counts.h5ad'

diseased_T_B_old_subAnno_meta_file_path = f'{obj}/cellphoneDB_db/samples_all_diseased_T_B_old_subAnno_meta_file.tsv'
diseased_T_B_old_degs_file_path = f'{obj}/cellphoneDB_db/diseased_T_B_old_subAnno_DEGs_file.tsv'
diseased_T_B_old_counts_file_path = f'{obj}/cellphoneDB_db/samples_all_diseased_T_B_old_normalised_log_counts.h5ad'

#healthy
healthy_T_B_young_subAnno_meta_file_path = f'{obj}/cellphoneDB_db/samples_all_healthy_T_B_young_subAnno_meta_file.tsv'
healthy_T_B_young_degs_file_path = f'{obj}/cellphoneDB_db/healthy_T_B_young_subAnno_DEGs_file.tsv'
healthy_T_B_young_counts_file_path = f'{obj}/cellphoneDB_db/samples_all_healthy_T_B_young_normalised_log_counts.h5ad'

healthy_T_B_old_subAnno_meta_file_path = f'{obj}/cellphoneDB_db/samples_all_healthy_T_B_old_subAnno_meta_file.tsv'
healthy_T_B_old_degs_file_path = f'{obj}/cellphoneDB_db/healthy_T_B_old_subAnno_DEGs_file.tsv'
healthy_T_B_old_counts_file_path = f'{obj}/cellphoneDB_db/samples_all_healthy_T_B_old_normalised_log_counts.h5ad'


### DEGs analysis (Method 3)

In [ ]:
from cellphonedb.src.core.methods import cpdb_degs_analysis_method

In [ ]:
#method3;diseased;young;T_B_subAnno
cpdb_diseased_T_B_young_results_deg_analysis = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = diseased_T_B_young_subAnno_meta_file_path,      # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = diseased_T_B_young_counts_file_path,             # mandatory: normalized count matrix.
    degs_file_path = diseased_T_B_young_degs_file_path, 
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                 # optional: whether to score interactions or not.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 30,                                     # number of threads to use in the analysis.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = f'{obj}/cellphoneDB_db',                          # Path to save results.
    output_suffix = 'samples_all_diseased_T_B_young_subAnno_cpdb_method3'                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
#method3;diseased;old;T_B_subAnno
cpdb_diseased_T_B_old_results_deg_analysis = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = diseased_T_B_old_subAnno_meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = diseased_T_B_old_counts_file_path,             # mandatory: normalized count matrix.
    degs_file_path = diseased_T_B_old_degs_file_path, 
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                 # optional: whether to score interactions or not.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 30,                                     # number of threads to use in the analysis.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = f'{obj}/cellphoneDB_db',                          # Path to save results.
    output_suffix = 'samples_all_diseased_T_B_old_subAnno_cpdb_method3'                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
#method3;healthy;young;T_B_subAnno
cpdb_healthy_T_B_young_results_deg_analysis = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = healthy_T_B_young_subAnno_meta_file_path,      # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = healthy_T_B_young_counts_file_path,             # mandatory: normalized count matrix.
    degs_file_path = healthy_T_B_young_degs_file_path, 
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                 # optional: whether to score interactions or not.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 30,                                     # number of threads to use in the analysis.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = f'{obj}/cellphoneDB_db',                          # Path to save results.
    output_suffix = 'samples_all_healthy_T_B_young_subAnno_cpdb_method3'                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )

In [ ]:
#method3;healthy;old;T_B_subAnno
cpdb_healthy_T_B_old_results_deg_analysis = cpdb_degs_analysis_method.call(
    cpdb_file_path = cpdb_file_path,                 # mandatory: CellphoneDB database zip file.
    meta_file_path = healthy_T_B_old_subAnno_meta_file_path,                 # mandatory: tsv file defining barcodes to cell label.
    counts_file_path = healthy_T_B_old_counts_file_path,             # mandatory: normalized count matrix.
    degs_file_path = healthy_T_B_old_degs_file_path, 
    counts_data = 'hgnc_symbol',                     # defines the gene annotation in counts matrix.
    score_interactions = True,                 # optional: whether to score interactions or not.
    threshold = 0.05,                                 # defines the min % of cells expressing a gene for this to be employed in the analysis.
    threads = 30,                                     # number of threads to use in the analysis.
    result_precision = 3,                            # Sets the rounding for the mean values in significan_means.
    separator = '|',                                 # Sets the string to employ to separate cells in the results dataframes "cellA|CellB".
    debug = False,                                   # Saves all intermediate tables employed during the analysis in pkl format.
    output_path = f'{obj}/cellphoneDB_db',                          # Path to save results.
    output_suffix = 'samples_all_healthy_T_B_old_subAnno_cpdb_method3'                             # Replaces the timestamp in the output files by a user defined string in the  (default: None).
    )